### Task 5: Model Iterations - Model 1 Logistic Regression

Victoria Vicheva (233182)

Team 9 

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np

1. Loading the dataset:

In [ ]:
# Load datasets (update paths as needed)
file_paths = [
    "D:\YearTwoAI\Block C\2024-25c-fai2-adsai-VictoriaVicheva233182\Week 2\clean_data.csv"
]
dfs = [pd.read_csv(file) for file in file_paths]
df = pd.concat(dfs, ignore_index=True)

2. Defining the core emotions (optional for this datset)

In [24]:
# Define core emotion mapping
core_emotions = {
    "happiness": [
        "amusement",
        "approval",
        "excitement",
        "gratitude",
        "joy",
        "love",
        "optimism",
        "pride",
        "relief",
    ],
    "sadness": ["grief", "remorse", "sadness"],
    "surprise": ["realization", "surprise"],
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "neutral": ["neutral"],
}

In [25]:
# Function to map emotions to core categories
def map_to_core_emotions(row):
    for core, emotions in core_emotions.items():
        if any(row[emotion] == 1 for emotion in emotions):
            return core
    return "neutral"


df["core_emotion"] = df.apply(map_to_core_emotions, axis=1)

In [26]:
# Select only text and core emotions
df_clean = df[["text", "core_emotion"]].dropna()

# Split data
X = df_clean["text"]
y = df_clean["core_emotion"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [27]:
# Convert text to numerical features using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [29]:
from collections import Counter

# Check class distribution before resampling
class_counts = Counter(y_train)
print("Original class distribution:", class_counts)

Original class distribution: Counter({'neutral': 75156, 'happiness': 50952, 'anger': 21851, 'surprise': 8817, 'sadness': 6867, 'disgust': 2673, 'fear': 2664})


In [30]:
# Define SMOTE strategy (only for minority classes)
smote_strategy = {"disgust": 5000, "fear": 5000, "surprise": 12000, "sadness": 10000}

In [31]:
# Apply SMOTE for minority class oversampling
smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)

In [32]:
# Define undersampling strategy (only for majority classes)
undersample_strategy = {"neutral": 10000, "happiness": 7000}

In [34]:
# Train Logistic Regression model
log_reg = LogisticRegression(max_iter=200, solver="saga", multi_class="multinomial")
log_reg.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=200, multi_class='multinomial', solver='saga')

In [35]:
# Make predictions
y_pred = log_reg.predict(X_test_tfidf)

In [36]:
# Evaluate model performance
classification_results = classification_report(y_test, y_pred)
print("Classification Report:\n", classification_results)

Classification Report:
               precision    recall  f1-score   support

       anger       0.47      0.23      0.31      5463
     disgust       0.43      0.08      0.13       668
        fear       0.48      0.22      0.30       666
   happiness       0.70      0.54      0.61     12738
     neutral       0.56      0.82      0.66     18789
     sadness       0.60      0.31      0.41      1717
    surprise       0.46      0.11      0.18      2204

    accuracy                           0.58     42245
   macro avg       0.53      0.33      0.37     42245
weighted avg       0.58      0.58      0.55     42245

